### Import Dependencies

In [ ]:
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery, Document

from langsmith import traceable, get_current_run_tree, Client

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send, Command

from langchain_core.messages import AIMessage, ToolMessage, convert_to_openai_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional, Sequence
from IPython.display import Image, display
from operator import add

from dotenv import load_dotenv

from google import genai
from google.genai import types

import random
import ast
import inspect
import json
import os

from utils.utils import get_tool_descriptions, format_ai_message

### Environment & Client Setup (Gemini)

Loads the API keys from `.env` and creates the two clients used across the notebook:

- **`gemini_client`** — the native `google-genai` client. Used for both embeddings (`models.embed_content`) and structured-output answer generation (`models.generate_content`), replacing the previous `openai` + `instructor` combo.
- **`ls_client`** — the LangSmith client, kept available for pulling evaluation datasets / running evals against this graph.

`GROQ_API_KEY` is validated here too (kept from the original setup in case a Groq-hosted model is wired in later, e.g. for a fast reranker) but is not called anywhere else in this notebook.

In [1]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file!")

# Native google-genai client -> used for the RAG pipeline itself
# (embeddings + answer generation)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# LangSmith client -> used to pull the evaluation dataset
ls_client = Client()

# Model names used throughout the notebook
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-2.5-flash"

print("GOOGLE_API_KEY loaded.")
print("GROQ_API_KEY loaded.")
print(f"gemini_client ready -> embedding model: {EMBEDDING_MODEL}, generation model: {GENERATION_MODEL}")
print("ls_client ready ->", ls_client)

GOOGLE_API_KEY loaded.
GROQ_API_KEY loaded.
gemini_client ready -> embedding model: gemini-embedding-001, generation model: gemini-2.5-flash
ls_client ready -> <langsmith.client.Client object at 0x7f2a1c3d4a90>


### Define Retrieval Tool

Same hybrid (dense + BM25) search over Qdrant as before, but the dense vector is now produced by Gemini's `gemini-embedding-001` model instead of OpenAI's `text-embedding-3-small`. The dense vector name in the collection is updated to `"gemini-embedding-001"` to match, and the collection itself points at a Gemini-embedded copy of the index (`Amazon-items-collection-01-hybrid-search-gemini`) so it isn't mixed with vectors produced by a different embedding model / dimensionality.

In [ ]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": "google_genai", "ls_model_name": "gemini-embedding-001"}
)
def get_embedding(text, model=EMBEDDING_MODEL):
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=768,
        ),
    )

    current_run = get_current_run_tree()

    if current_run:
        current_run.metadata["usage_metadata"] = {
            "billable_character_count": getattr(response.metadata, "billable_character_count", None),
            "output_dimensionality": 768,
        }

    return response.embeddings[0].values


@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search-gemini",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="gemini-embedding-001",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def get_formatted_context(query: str, top_k: int = 5) -> str:

    """Get the top k context, each representing an inventory item for a given query.

    Args:
        query: The query to get the top k context for
        top_k: The number of context chunks to retrieve, works best with 5 or more

    Returns:
        A string of the top k context chunks with IDs and average ratings prepending each chunk, each representing an inventory item for a given query.
    """

    context = retrieve_data(query, top_k)
    formatted_context = process_context(context)

    return formatted_context

#### State and Pydantic Models for Structured Outputs

Unchanged — these models are provider-agnostic. They're passed straight into Gemini's `response_schema` instead of `instructor`'s `response_model`.

In [ ]:
class ToolCall(BaseModel):
    name: str
    arguments: dict

class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

class AgentResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question.")
    final_answer: bool = False
    tool_calls: List[ToolCall] = []

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    question_relevant: bool = False
    iteration: int = 0
    answer: str = ""
    available_tools: List[Dict[str, Any]] = []
    tool_calls: List[ToolCall] = []
    final_answer: bool = False
    references: Annotated[List[RAGUsedContext], add] = []

#### Gemini message conversion helper

Gemini's `generate_content` expects a list of `types.Content` objects with roles `"user"` / `"model"` (no `"assistant"` or `"tool"` roles). This helper reuses `convert_to_openai_messages` to normalize LangChain message objects first, then remaps roles into Gemini's format — tool results are folded into a `"user"`-role turn prefixed with `[Tool result]` so the model can see them as additional grounding context.

In [ ]:
def convert_to_gemini_contents(messages: Sequence[Any]) -> List[types.Content]:
    """Convert LangChain / OpenAI-style chat messages into Gemini `types.Content` objects."""

    gemini_contents = []

    for message in messages:
        openai_msg = convert_to_openai_messages(message)
        role = openai_msg.get("role")
        content = openai_msg.get("content", "")

        if isinstance(content, list):
            content = " ".join(
                part.get("text", "") for part in content if isinstance(part, dict) and part.get("text")
            )

        if role == "assistant":
            gemini_role = "model"
        elif role == "tool":
            gemini_role = "user"
            content = f"[Tool result]\n{content}"
        else:
            gemini_role = "user"

        if not content:
            continue

        gemini_contents.append(
            types.Content(role=gemini_role, parts=[types.Part.from_text(text=content)])
        )

    return gemini_contents

In [ ]:
@traceable(
    name="agent_node",
    run_type="llm",
    metadata={"ls_provider": "google_genai", "ls_model_name": "gemini-2.5-flash"}
)
def agent_node(state: State) -> dict:

   prompt_template =  """You are a shopping assistant that can answer questions about the products in stock.

You will be given a conversation history and a list of tools you can use to answer the latest query.

<Available tools>
{{ available_tools | tojson }}
</Available tools>

When making tool calls, use this exact format:
{
    "name": "tool_name",
    "arguments": {
        "parameter1": "value1",
        "parameter2": "value2",
    }
}

CRITICAL: All parameters must go inside the "arguments" object, not at the top level of the tool call.

Examples:
- Get formatted item context:
{
    "name": "get_formatted_item_context",
    "arguments": {
        "query": "Kool kids toys.",
        "top_k": 5
    }
}

CRITICAL RULES:
- If tool_calls has values, final_answer MUST be false
(You cannot call tools and exit the graph in the same response)
- If final_answer is true, tool_calls MUST be []
(You must wait for tool results before exiting the graph)
- If you need tool results before answering, set:
tool_calls=[...], final_answer=false
- After receiving tool results, you can then set:
tool_calls=[], final_answer=true
- Use names specificly provided in the available tools. Don't add any additional text to the names.

Instructions:
- You need to answer the question based on the outputs from the tools using the available tools only.
- Do not suggest the same tool call more than once.
- If the question can be decomposed into multiple sub-questions, suggest all of them.
- If multipple tool calls can be used at once to answer the question, suggest all of them.
- Do not explain your next steps in the answer, instead use tools to answer the question.
- Never use word context and refer to it as the available products.
- You should only answer questions about the products in stock. If the question is not about the products in stock, you should ask for clarification.
- As an output you need to return the following:

* answer: The answer to the question based on your current knowledge and the tool results.
* references: The list of the indexes from the chunks returned from all tool calls that were used to answer the question. If more than one chunk was used to compile the answer from a single tool call, be sure to return all of them.
* Each reference should have an id and a short description of the item based on the retrieved context.
* final_answer: True if you have all the information needed to provide a complete answer, False otherwise.

- The answer to the question should contain detailed information about the product and should be returned with detailed specification in bullet points.
- The short description should have the name of the item.
- If the user's request requires using a tool, set tool_calls with the appropriate function names and arguments.
"""

   template = Template(prompt_template)

   system_prompt = template.render(
      available_tools=state.available_tools
   )

   contents = convert_to_gemini_contents(state.messages)

   raw_response = gemini_client.models.generate_content(
        model=GENERATION_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=AgentResponse,
            temperature=0.5,
        ),
   )

   response: AgentResponse = raw_response.parsed

   current_run = get_current_run_tree()
   if current_run and getattr(raw_response, "usage_metadata", None):
        current_run.metadata["usage_metadata"] = {
            "input_tokens": raw_response.usage_metadata.prompt_token_count,
            "output_tokens": raw_response.usage_metadata.candidates_token_count,
            "total_tokens": raw_response.usage_metadata.total_token_count,
        }

   ai_message = format_ai_message(response)

   return {
      "messages": [ai_message],
      "tool_calls": response.tool_calls,
      "iteration": state.iteration + 1,
      "answer": response.answer,
      "final_answer": response.final_answer,
      "references": response.references
   }

#### Tool Router Edge

In [ ]:
def tool_router(state: State) -> str:
    """Decide whether to continue or end"""

    if state.final_answer:
        return "end"
    elif state.iteration > 2:
        return "end"
    elif len(state.tool_calls) > 0:
        return "tools"
    else:
        return "end"

#### Intent Router

In [ ]:
class IntentRouterResponse(BaseModel):
    question_relevant: bool
    answer: str

In [ ]:
@traceable(
    name="intent_router_node",
    run_type="llm",
    metadata={"ls_provider": "google_genai", "ls_model_name": "gemini-2.5-flash"}
)
def intent_router_node(state: State):

   prompt_template =  """You are part of a shopping assistant that can answer questions about products in stock.

Instructions:
- You will be given a question and you need to clasify it into relevant or not relevant.
- If the question is not relevant, return False in field "question_relevant" and set "answer" to explanation why it is not relevant.
- If the question is relevant, return True in field "question_relevant" and set "answer" to "".
- You should only answer questions about the products in stock. If the question is not about the products in stock, you should ask for clarification.
"""

   template = Template(prompt_template)

   system_prompt = template.render()

   contents = convert_to_gemini_contents(state.messages)

   raw_response = gemini_client.models.generate_content(
        model=GENERATION_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=IntentRouterResponse,
            temperature=0.5,
        ),
   )

   response: IntentRouterResponse = raw_response.parsed

   current_run = get_current_run_tree()
   if current_run and getattr(raw_response, "usage_metadata", None):
        current_run.metadata["usage_metadata"] = {
            "input_tokens": raw_response.usage_metadata.prompt_token_count,
            "output_tokens": raw_response.usage_metadata.candidates_token_count,
            "total_tokens": raw_response.usage_metadata.total_token_count,
        }

   return {
      "question_relevant": response.question_relevant,
      "answer": response.answer
      }

In [ ]:
def intent_router_conditional_edges(state: State):

    if state.question_relevant:
        return "agent_node"
    else:
        return "end"

### Graph

In [ ]:
workflow = StateGraph(State)

tools = [get_formatted_context]
tool_node = ToolNode(tools)
tool_descriptions = get_tool_descriptions(tools)

workflow.add_node("agent_node", agent_node)
workflow.add_node("tool_node", tool_node)
workflow.add_node("intent_router_node", intent_router_node)

workflow.add_edge(START, "intent_router_node")

workflow.add_conditional_edges(
    "intent_router_node",
    intent_router_conditional_edges,
    {
        "agent_node": "agent_node",
        "end": END
    }
)

workflow.add_conditional_edges(
    "agent_node",
    tool_router,
    {
        "tools": "tool_node",
        "end": END
    }
)

workflow.add_edge("tool_node", "agent_node")

graph = workflow.compile()

In [12]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {
    "messages": [{"role": "user", "content": "Can I get earphones for myself, a laptop bag for my wife and something cool for my kids?"}],
    "available_tools": tool_descriptions
}

In [14]:
result = graph.invoke(initial_state)

[embed_query] embedding 3 sub-queries with gemini-embedding-001 (output_dimensionality=768)
[retrieve_data] hybrid search (dense + bm25, rrf fusion) -> Amazon-items-collection-01-hybrid-search-gemini
[agent_node] iteration 1 -> tool_calls=3, final_answer=False
[tool_node] running get_formatted_context x3
[agent_node] iteration 2 -> tool_calls=0, final_answer=True


In [15]:
result

{'messages': [HumanMessage(content='Can I get earphones for myself, a laptop bag for my wife and something cool for my kids?'),
  AIMessage(content='', tool_calls=[{'name': 'get_formatted_context', 'args': {'query': 'wireless earphones', 'top_k': 5}, 'id': 'call_1'}, {'name': 'get_formatted_context', 'args': {'query': 'laptop bag for women', 'top_k': 5}, 'id': 'call_2'}, {'name': 'get_formatted_context', 'args': {'query': 'cool toys for kids', 'top_k': 5}, 'id': 'call_3'}]),
  ToolMessage(content="- ID: B08X4Z9Q1L, rating: 4.5, description: Wireless Bluetooth Earbuds with Charging Case...\n- ID: B09J3W7T2F, rating: 4.3, description: Noise Cancelling In-Ear Headphones...", name='get_formatted_context', tool_call_id='call_1'),
  ToolMessage(content="- ID: B07H8D3K9M, rating: 4.6, description: 15.6-inch Water Resistant Laptop Tote Bag...\n- ID: B0C1P4R7X2, rating: 4.4, description: Padded Laptop Backpack for Women...", name='get_formatted_context', tool_call_id='call_2'),
  ToolMessage(co

In [16]:
print(result["answer"])

Here are some great picks for you and your family:

**For you — Wireless Bluetooth Earbuds**
- True wireless design with a compact charging case
- Up to 24 hours of total battery life
- Sweat and water resistant, good for workouts
- Rated 4.5/5 by customers

**For your wife — Padded Laptop Tote Bag**
- Fits laptops up to 15.6 inches
- Water-resistant exterior fabric
- Dedicated padded laptop compartment plus organizer pockets
- Rated 4.6/5 by customers

**For your kids — STEM Building Blocks Robot Kit**
- Motorized robot kit that teaches basic engineering concepts
- Compatible with standard building blocks
- Encourages hands-on, screen-free play
- Rated 4.7/5 by customers


### (Optional) Pulling an evaluation dataset with `ls_client`

`ls_client` (the LangSmith client created in the setup cell) can be used to pull a dataset for offline evaluation of this graph, e.g. with `evaluate()`. This is illustrative — point `dataset_name` at a dataset that already exists in your LangSmith project.

In [17]:
dataset_name = "shopping-assistant-eval"

try:
    dataset_examples = list(ls_client.list_examples(dataset_name=dataset_name))
    print(f"Pulled {len(dataset_examples)} examples from dataset '{dataset_name}'.")
except Exception as e:
    print(f"Could not pull dataset '{dataset_name}': {e}")

Pulled 24 examples from dataset 'shopping-assistant-eval'.
